# Wan2.1-T2V-1.3B BSS/BDS Colab Run-All

Use Colab `Runtime > Run all` on this notebook.

This notebook keeps the large Wan2.1-T2V-1.3B checkpoint on Google Drive, stages it to local Colab disk for speed, pulls the latest experiment code from Git, then runs the Wan2.1 same-compute BSS/BDS smoke or full mini-suite.

Default run is `RUN_SCALE = "smoke"`. Change it to `"full"` after smoke passes to run the 8-prompt suite. Wan2.2 is audited only as a future extension and is not run here.

In [8]:
from pathlib import Path
import os

REPO_URL = "https://github.com/WANG-Ruipeng/Wan2.1.git"
BRANCH = "main"
REPO_DIR = Path("/content/Wan2.1")

EXPERIMENT_NAME = "wan21_13b_bss_bds_v1"
LOCAL_EXP_ROOT = REPO_DIR / "bss_experiments" / EXPERIMENT_NAME
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS")
DRIVE_EXP_ROOT = DRIVE_PROJECT_ROOT / EXPERIMENT_NAME

DRIVE_MODEL_DIR = DRIVE_PROJECT_ROOT / "models/Wan-AI/Wan2.1-T2V-1.3B"
LOCAL_MODEL_DIR = Path("/content/models/Wan-AI/Wan2.1-T2V-1.3B")
HF_MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B"
MODELSCOPE_MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B"
MODEL_DOWNLOAD_PROVIDER = "huggingface"  # huggingface or modelscope

RUN_SCALE = os.environ.get("WAN21_BSS_RUN_SCALE", "full")  # smoke, full, analysis_only
RUN_LIMIT = int(os.environ.get("WAN21_BSS_RUN_LIMIT", "0"))  # 0 means no limit

MOUNT_DRIVE = True
INSTALL_DEPS = True
INSTALL_FLASH_ATTN = False  # SDPA fallback is used when flash-attn is unavailable
DOWNLOAD_MODEL_IF_MISSING = True
FORCE_RECLONE = False
FORCE_RERUN_ROWS = False
FORCE_RESTAGE_MODEL = False
USE_MEMORY_SAVING_FLAGS = True
OOM_RETRY_MEMORY_SAVING = True
SYNC_RESULTS_TO_DRIVE = True
SYNC_HEAVY = True

print("RUN_SCALE:", RUN_SCALE)
print("Repo:", REPO_URL, "branch", BRANCH)
print("Drive model cache:", DRIVE_MODEL_DIR)
print("Local model staging:", LOCAL_MODEL_DIR)

RUN_SCALE: full
Repo: https://github.com/WANG-Ruipeng/Wan2.1.git branch main
Drive model cache: /content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS/models/Wan-AI/Wan2.1-T2V-1.3B
Local model staging: /content/models/Wan-AI/Wan2.1-T2V-1.3B


In [9]:
import getpass
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped or failed:", repr(exc))

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
print("Runtime setup done.")


def display_cmd(cmd):
    text = " ".join(str(part) for part in cmd)
    if "x-access-token:" in text:
        text = text.split("x-access-token:")[0] + "x-access-token:***@" + text.split("@", 1)[-1]
    return text


def run(cmd, cwd=None, check=True):
    print("$", display_cmd(cmd))
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=check, text=True)


def run_capture(cmd, cwd=None, check=False):
    print("$", display_cmd(cmd))
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=check, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)


def authenticated_url(url):
    token = os.environ.get("GITHUB_TOKEN")
    if token is None:
        token = getpass.getpass("GitHub token for clone/fetch; leave blank if the branch is public: ")
        if token:
            os.environ["GITHUB_TOKEN"] = token
    if token:
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime setup done.


In [10]:
clone_url = authenticated_url(REPO_URL)

if FORCE_RECLONE and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    fallback = REPO_DIR.with_name(f"{REPO_DIR.name}_non_git_{int(time.time())}")
    print(f"Existing non-git directory found at {REPO_DIR}; moving it to {fallback}")
    shutil.move(str(REPO_DIR), str(fallback))

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", clone_url, str(REPO_DIR)])
else:
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", clone_url])
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH])
    switched = run(["git", "-C", str(REPO_DIR), "switch", BRANCH], check=False)
    if switched.returncode != 0:
        run(["git", "-C", str(REPO_DIR), "switch", "--track", "-c", BRANCH, f"origin/{BRANCH}"])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH])

print(run_capture(["git", "-C", str(REPO_DIR), "status", "--short"]).stdout)
print(run_capture(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"]).stdout)
REQUIRED_REPO_FILES = [
    "wan/utils/schedule_control.py",
    "scripts/audit_wan21_13b_bss_bds.py",
    "scripts/make_manifest_wan21_13b_bds.py",
    "scripts/run_manifest_wan21_13b_bds.py",
    "scripts/validate_schedules_wan21_13b_bds.py",
    "scripts/compute_metrics_against_ref.py",
    "scripts/compute_bds.py",
    "scripts/make_final_outputs_wan21_13b_bds.py",
]
missing_repo_files = [name for name in REQUIRED_REPO_FILES if not (REPO_DIR / name).exists()]
if missing_repo_files:
    raise FileNotFoundError(
        "The cloned Git branch does not contain the Wan2.1 BSS/BDS scripts. "
        "Push the local changes to the branch configured in the first cell, then rerun. Missing: "
        + ", ".join(missing_repo_files)
    )
print("Required BSS/BDS repo files are present.")


$ git -C /content/Wan2.1 remote set-url origin https://github.com/WANG-Ruipeng/Wan2.1.git
$ git -C /content/Wan2.1 fetch origin main
$ git -C /content/Wan2.1 switch main
$ git -C /content/Wan2.1 pull --ff-only origin main
$ git -C /content/Wan2.1 status --short
 M bss_experiments/wan21_13b_bss_bds_v1/manifests/wan21_13b_smoke_manifest.csv
 M bss_experiments/wan21_13b_bss_bds_v1/reports/00_repo_model_hardware_audit.md
 M bss_experiments/wan21_13b_bss_bds_v1/reports/00b_wan22_future_extension_audit.md

$ git -C /content/Wan2.1 rev-parse HEAD
5a3f312b7e3b87ca3c45f3b8ac444a3462ecaa98

Required BSS/BDS repo files are present.


In [11]:
if INSTALL_DEPS:
    req_path = REPO_DIR / "requirements.txt"
    filtered_req = Path("/tmp/wan21_requirements_no_flash.txt")
    lines = []
    for line in req_path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped in {"flash_attn", "flash-attn"}:
            continue
        lines.append(line)
    filtered_req.write_text("\n".join(lines) + "\n", encoding="utf-8")
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered_req)])
    run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[cli]", "modelscope"])
    if INSTALL_FLASH_ATTN:
        rc = run([sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"], check=False)
        if rc.returncode != 0:
            print("WARNING: flash-attn install failed. Checkpoint download/cache can continue, but inference may need flash-attn or a fallback patch.")
    print("Dependency install/check completed.")
else:
    print("INSTALL_DEPS=False; skipping dependency install/check.")

$ /usr/bin/python3 -m pip install -q -r /tmp/wan21_requirements_no_flash.txt
$ /usr/bin/python3 -m pip install -q huggingface_hub[cli] modelscope
Dependency install/check completed.


In [12]:
WAN_REQUIRED_FILES = [
    "diffusion_pytorch_model.safetensors",
    "models_t5_umt5-xxl-enc-bf16.pth",
    "Wan2.1_VAE.pth",
]


def wan_dir_valid(path):
    path = Path(path)
    return path.is_dir() and all((path / name).exists() for name in WAN_REQUIRED_FILES)


def rsync_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    if shutil.which("rsync"):
        run(["rsync", "-a", "--info=progress2", str(src) + "/", str(dst) + "/"])
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)


if not wan_dir_valid(DRIVE_MODEL_DIR) and DOWNLOAD_MODEL_IF_MISSING:
    print("Drive checkpoint cache missing; downloading once to Drive.")

    def download_from_huggingface():
        return run_capture(["huggingface-cli", "download", HF_MODEL_ID, "--local-dir", str(DRIVE_MODEL_DIR)], check=False)

    def download_from_modelscope():
        return run_capture(["modelscope", "download", MODELSCOPE_MODEL_ID, "--local_dir", str(DRIVE_MODEL_DIR)], check=False)

    providers = [MODEL_DOWNLOAD_PROVIDER]
    if MODEL_DOWNLOAD_PROVIDER == "huggingface":
        providers.append("modelscope")
    elif MODEL_DOWNLOAD_PROVIDER == "modelscope":
        providers.append("huggingface")

    last_output = ""
    for provider in providers:
        print(f"Trying model download provider: {provider}")
        result = download_from_huggingface() if provider == "huggingface" else download_from_modelscope()
        last_output = result.stdout or ""
        print(last_output)
        if result.returncode == 0 and wan_dir_valid(DRIVE_MODEL_DIR):
            print(f"Download succeeded with {provider}.")
            break
        print(f"Download provider {provider} failed or produced an incomplete checkpoint.")
    if not wan_dir_valid(DRIVE_MODEL_DIR):
        raise RuntimeError("Checkpoint download failed. Last downloader output:\n" + last_output[-4000:])

if not wan_dir_valid(DRIVE_MODEL_DIR):
    print("Missing required checkpoint files in Drive cache:")
    for name in WAN_REQUIRED_FILES:
        print(name, "OK" if (DRIVE_MODEL_DIR / name).exists() else "MISSING")
    raise FileNotFoundError(f"Wan2.1-T2V-1.3B checkpoint not ready at {DRIVE_MODEL_DIR}")

if FORCE_RESTAGE_MODEL and LOCAL_MODEL_DIR.exists():
    shutil.rmtree(LOCAL_MODEL_DIR)

if not wan_dir_valid(LOCAL_MODEL_DIR):
    print("Staging checkpoint from Drive to local Colab disk.")
    rsync_or_copy(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
else:
    print("Local checkpoint staging already valid.")

print("Drive checkpoint valid:", wan_dir_valid(DRIVE_MODEL_DIR), DRIVE_MODEL_DIR)
print("Local checkpoint valid:", wan_dir_valid(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR)

Local checkpoint staging already valid.
Drive checkpoint valid: True /content/drive/MyDrive/Colab_Projects/Wan2.1-BSS-BDS/models/Wan-AI/Wan2.1-T2V-1.3B
Local checkpoint valid: True /content/models/Wan-AI/Wan2.1-T2V-1.3B


In [13]:
if DRIVE_EXP_ROOT.exists() and not LOCAL_EXP_ROOT.exists():
    print(f"Restoring previous experiment artifacts from Drive: {DRIVE_EXP_ROOT} -> {LOCAL_EXP_ROOT}")
    rsync_or_copy(DRIVE_EXP_ROOT, LOCAL_EXP_ROOT)
else:
    print("No Drive restore needed, or local experiment folder already exists.")

LOCAL_EXP_ROOT.mkdir(parents=True, exist_ok=True)

No Drive restore needed, or local experiment folder already exists.


In [14]:
print("===== GPU =====")
print(run_capture(["nvidia-smi"]).stdout)

run([sys.executable, str(REPO_DIR / "scripts/audit_wan21_13b_bss_bds.py"),
     "--repo_dir", str(REPO_DIR),
     "--output_root", str(LOCAL_EXP_ROOT.relative_to(REPO_DIR)),
     "--ckpt_dir", str(LOCAL_MODEL_DIR)])

run([sys.executable, str(REPO_DIR / "scripts/make_manifest_wan21_13b_bds.py"),
     "--repo_dir", str(REPO_DIR),
     "--output_root", str(LOCAL_EXP_ROOT.relative_to(REPO_DIR))])

SMOKE_MANIFEST = LOCAL_EXP_ROOT / "manifests/wan21_13b_smoke_manifest.csv"
FULL_MANIFEST = LOCAL_EXP_ROOT / "manifests/wan21_13b_prompt_suite_manifest.csv"
print("Smoke manifest:", SMOKE_MANIFEST)
print("Full manifest:", FULL_MANIFEST)

===== GPU =====
$ nvidia-smi
Sun Jun 28 12:47:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------------

In [ ]:
def tail_text(path, max_chars=8000):
    path = Path(path)
    if not path.exists():
        return f"<missing: {path}>"
    return path.read_text(encoding="utf-8", errors="replace")[-max_chars:]


def print_failed_manifest_logs(manifest_path):
    import csv
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        print("Manifest missing:", manifest_path)
        return
    rows = list(csv.DictReader(manifest_path.open(newline="", encoding="utf-8")))
    problem_rows = [row for row in rows if row.get("status") in {"failed", "running"}]
    if not problem_rows:
        print("No failed/running row found in manifest yet.")
        return
    for row in problem_rows[:3]:
        print("\n===== Failed manifest row =====")
        print("run_id:", row.get("run_id"))
        print("method:", row.get("method"))
        print("error_message:", row.get("error_message"))
        for label, field in [("stdout", "stdout_log_path"), ("stderr", "stderr_log_path"), ("runtime", "runtime_json_path")]:
            path = REPO_DIR / row[field] if not Path(row[field]).is_absolute() else Path(row[field])
            print(f"\n----- tail {label}: {path} -----")
            print(tail_text(path))


def run_manifest(manifest_path):
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts/run_manifest_wan21_13b_bds.py"),
        "--repo_dir", str(REPO_DIR),
        "--manifest", str(manifest_path),
        "--ckpt_dir", str(LOCAL_MODEL_DIR),
        "--resume",
    ]
    if RUN_LIMIT:
        cmd.extend(["--limit", str(RUN_LIMIT)])
    if FORCE_RERUN_ROWS:
        cmd.append("--force")
    if USE_MEMORY_SAVING_FLAGS:
        cmd.extend(["--offload_model", "--t5_cpu"])
    if OOM_RETRY_MEMORY_SAVING:
        cmd.append("--oom_retry_memory_saving")
    try:
        run(cmd, cwd=REPO_DIR)
    except subprocess.CalledProcessError:
        print_failed_manifest_logs(manifest_path)
        raise


def validate_and_analyze(manifest_path, report_name):
    run([sys.executable, str(REPO_DIR / "scripts/validate_schedules_wan21_13b_bds.py"),
         "--repo_dir", str(REPO_DIR),
         "--manifest", str(manifest_path),
         "--output_root", str(LOCAL_EXP_ROOT.relative_to(REPO_DIR)),
         "--report_name", report_name])
    run([sys.executable, str(REPO_DIR / "scripts/compute_metrics_against_ref.py"),
         "--repo_dir", str(REPO_DIR),
         "--manifest", str(manifest_path),
         "--output_root", str(LOCAL_EXP_ROOT.relative_to(REPO_DIR))])
    run([sys.executable, str(REPO_DIR / "scripts/compute_bds.py"),
         "--repo_dir", str(REPO_DIR),
         "--output_root", str(LOCAL_EXP_ROOT.relative_to(REPO_DIR))])
    run([sys.executable, str(REPO_DIR / "scripts/make_final_outputs_wan21_13b_bds.py"),
         "--repo_dir", str(REPO_DIR),
         "--output_root", str(LOCAL_EXP_ROOT.relative_to(REPO_DIR))])


if RUN_SCALE in {"smoke", "full"}:
    run_manifest(SMOKE_MANIFEST)
    validate_and_analyze(SMOKE_MANIFEST, "01_wan21_13b_smoke_report.md")
else:
    print("Skipping smoke run because RUN_SCALE=analysis_only")

$ /usr/bin/python3 /content/Wan2.1/scripts/run_manifest_wan21_13b_bds.py --repo_dir /content/Wan2.1 --manifest /content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/manifests/wan21_13b_smoke_manifest.csv --ckpt_dir /content/models/Wan-AI/Wan2.1-T2V-1.3B --resume --offload_model --t5_cpu --oom_retry_memory_saving

===== Failed manifest row =====
run_id: smoke_official_cats_bss10
method: bss10
error_message: Traceback (most recent call last):
  File "/content/Wan2.1/generate.py", line 672, in <module>
    generate(args)
  File "/content/Wan2.1/generate.py", line 453, in generate
    video = wan_t2v.generate(
            ^^^^^^^^^^^^^^^^^
  File "/content/Wan2.1/wan/text2video.py", line 243, in generate
    timesteps, bss_info = configure_bss_scheduler(
                          ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Wan2.1/wan/utils/schedule_control.py", line 109, in configure_bss_scheduler
    sample_scheduler.set_timesteps(
  File "/content/Wan2.1/wan/utils/fm_solvers_unipc.py", line

CalledProcessError: Command '['/usr/bin/python3', '/content/Wan2.1/scripts/run_manifest_wan21_13b_bds.py', '--repo_dir', '/content/Wan2.1', '--manifest', '/content/Wan2.1/bss_experiments/wan21_13b_bss_bds_v1/manifests/wan21_13b_smoke_manifest.csv', '--ckpt_dir', '/content/models/Wan-AI/Wan2.1-T2V-1.3B', '--resume', '--offload_model', '--t5_cpu', '--oom_retry_memory_saving']' returned non-zero exit status 1.

In [ ]:
if RUN_SCALE == "full":
    run_manifest(FULL_MANIFEST)
    validate_and_analyze(FULL_MANIFEST, "02_mini_suite_run_report.md")
elif RUN_SCALE == "analysis_only":
    manifest = FULL_MANIFEST if FULL_MANIFEST.exists() else SMOKE_MANIFEST
    validate_and_analyze(manifest, "02_mini_suite_run_report.md")
else:
    print("RUN_SCALE is smoke; full mini-suite not run. Set RUN_SCALE='full' after smoke passes.")

In [ ]:
if SYNC_RESULTS_TO_DRIVE:
    DRIVE_EXP_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which("rsync"):
        cmd = ["rsync", "-a", str(LOCAL_EXP_ROOT) + "/", str(DRIVE_EXP_ROOT) + "/"]
        if not SYNC_HEAVY:
            cmd[2:2] = ["--exclude", "outputs/videos/"]
        run(cmd)
    else:
        shutil.copytree(LOCAL_EXP_ROOT, DRIVE_EXP_ROOT, dirs_exist_ok=True)
    print("Synced experiment artifacts to Drive:", DRIVE_EXP_ROOT)
else:
    print("SYNC_RESULTS_TO_DRIVE=False; skipping Drive sync.")

In [ ]:
important_paths = {
    "experiment folder": LOCAL_EXP_ROOT,
    "Drive mirror": DRIVE_EXP_ROOT,
    "audit report": LOCAL_EXP_ROOT / "reports/00_repo_model_hardware_audit.md",
    "Wan2.2 audit": LOCAL_EXP_ROOT / "reports/00b_wan22_future_extension_audit.md",
    "smoke report": LOCAL_EXP_ROOT / "reports/01_wan21_13b_smoke_report.md",
    "mini-suite report": LOCAL_EXP_ROOT / "reports/02_mini_suite_run_report.md",
    "BDS table": LOCAL_EXP_ROOT / "tables/tableA_wan21_13b_bds_by_split.md",
    "paper row": LOCAL_EXP_ROOT / "tables/table_cross_model_same_compute_wan21_row.md",
    "LaTeX row": LOCAL_EXP_ROOT / "tables/table_cross_model_same_compute_wan21_row.tex",
    "final report": LOCAL_EXP_ROOT / "reports/FINAL_WAN21_13B_BSS_BDS_REPORT.md",
}

for label, path in important_paths.items():
    print(f"{label:20s} {'OK' if path.exists() else 'not yet'}  {path}")

final_report = LOCAL_EXP_ROOT / "reports/FINAL_WAN21_13B_BSS_BDS_REPORT.md"
if final_report.exists():
    print("\n===== Final Report =====")
    print(final_report.read_text(encoding="utf-8"))
else:
    print("Final report not found yet.")